# Runiverse · ACE-Step 1.5 추론 시작하기
**Google Colab / 추론용 Python 3.11 / NVIDIA CUDA GPU**

런타임 메뉴에서 **런타임 유형 변경 → GPU**를 선택한 뒤 위에서 아래로 실행하세요.
기본 실험은 **Turbo · 30초 · 170 BPM · 무보컬 · 1개 · LM 미사용**입니다.

**Python 환경:** 코랩 노트북 커널을 강제로 바꾸지 않습니다. `uv`가 별도 Python 3.11과 가상환경을 만들고, 다운로드·모델 로딩·추론은 모두 그 환경의 Python으로 실행합니다. 코랩 커널은 설정·결과 표시만 담당합니다. 설치 후 런타임을 재시작할 필요가 없습니다.

처음에는 라이브러리와 수 GB의 모델 다운로드가 필요합니다. 코랩 GPU 할당·세션 유지가 보장되는 것은 아닙니다. 결과를 다운로드하거나 마지막 선택 셀로 Drive에 복사하세요.

**검증 범위:** 공식 소스의 함수·매개변수와 대조하고 노트북 구조·문법을 검증한 초안입니다. 실제 코랩 GPU에서 설치·음원 생성은 아직 실행하지 않았습니다.


## 1. GPU와 작업 경로 확인
CUDA가 표시된 `nvidia-smi` 출력은 드라이버 지원 정보를 뜻합니다. 설치 후 PyTorch에서 실제 CUDA 연산을 한 번 더 확인합니다.


In [ ]:
import sys
import os
import json
import subprocess
import shutil
from pathlib import Path

if not Path("/content").exists():
    raise RuntimeError("이 노트북은 Google Colab의 /content 환경을 기준으로 작성했습니다.")

ROOT = Path("/content/runiverse_acestep")
REPO = ROOT / "ACE-Step-1.5"
OUTPUT_ROOT = ROOT / "outputs"
ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# 확인 당시 공식 main의 커밋. 실험 재현을 위해 고정합니다.
SOURCE_COMMIT = "ca1e85fe9430179831e6bc6be790c332190a3866"
REPO_URL = "https://github.com/ace-step/ACE-Step-1.5.git"

print("코랩 커널 Python:", sys.version.split()[0])
print("추론용 Python은 다음 셀에서 별도 3.11로 설치합니다.")
try:
    gpu_check = subprocess.run(["nvidia-smi"], check=True, text=True, capture_output=True)
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    raise RuntimeError("GPU 런타임을 선택하고 다시 실행하세요.") from exc
print(gpu_check.stdout)
print(f"남은 디스크: {shutil.disk_usage('/content').free / 2**30:.1f} GiB")


## 2. Python 3.11과 공식 의존성 설치
공식 저장소의 `uv.lock`을 `--frozen`으로 사용합니다. 별도 환경이라 코랩 기본 PyTorch를 덮어쓰지 않습니다.
고정 소스는 Linux x86_64용 PyTorch 2.10.0 / CUDA 12.8 패키지를 지정합니다. 코랩 드라이버가 호환되어야 합니다.
설치 오류가 나면 로그를 확인하세요. 중간에 실패해도 이 셀을 다시 실행할 수 있습니다.


In [ ]:
def run_command(args, *, cwd=None, env=None):
    # 문자열 shell 대신 인자 배열로 실행합니다. 설치 로그는 바로 표시됩니다.
    print("실행:", " ".join(map(str, args)))
    subprocess.run(list(map(str, args)), cwd=cwd, env=env, check=True)

run_command([sys.executable, "-m", "pip", "install", "--quiet", "uv>=0.8"])
UV = shutil.which("uv")
if not UV:
    raise RuntimeError("uv 실행 파일을 찾지 못했습니다.")

run_command([UV, "python", "install", "3.11"])

if not REPO.exists():
    run_command(["git", "clone", "--no-checkout", "--depth", "1", REPO_URL, REPO])
elif not (REPO / ".git").exists():
    raise RuntimeError(f"{REPO}에 다른 파일이 있습니다. 별도 작업 경로를 사용하세요.")

run_command(["git", "fetch", "--depth", "1", "origin", SOURCE_COMMIT], cwd=REPO)
run_command(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO)
run_command([UV, "sync", "--frozen", "--no-dev", "--python", "3.11"], cwd=REPO)

PY311 = REPO / ".venv" / "bin" / "python"
if not PY311.exists():
    raise RuntimeError("추론용 Python 환경 생성에 실패했습니다.")

INFER_ENV = os.environ.copy()
INFER_ENV.update({
    "TOKENIZERS_PARALLELISM": "false",
    "ACESTEP_CHECKPOINTS_DIR": str(REPO / "checkpoints"),
    "HF_HOME": str(ROOT / "hf_cache"),
    "ACESTEP_INIT_LLM": "false",  # 첫 실험은 LM 없이 진행
})

def run_py311(script, *args):
    run_command([PY311, script, *args], cwd=REPO, env=INFER_ENV)

print("추론용 Python:", PY311)


In [ ]:
INFER_ENV["MPLBACKEND"] = "Agg"

## 3. Python 3.11 및 실제 CUDA 연산 확인
GPU가 없으면 CPU로 조용히 전환하지 않고 중단합니다. T4처럼 BF16을 지원하지 않는 GPU는 ACE-Step이 FP16을 선택합니다.


In [ ]:
gpu_probe = ROOT / "gpu_probe.py"
gpu_probe.write_text("""
import json
import platform
import sys
import torch
from pathlib import Path

assert sys.version_info[:2] == (3, 11), sys.version
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch에서 CUDA를 사용할 수 없습니다. GPU 런타임과 드라이버 호환성을 확인하세요.")
x = torch.ones((32, 32), device="cuda")
y = x @ x
torch.cuda.synchronize()
assert y.device.type == "cuda" and float(y[0, 0]) == 32.0
p = torch.cuda.get_device_properties(0)
info = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "gpu": p.name,
    "gpu_vram_gib": round(p.total_memory / 2**30, 2),
    "bf16_supported": bool(torch.cuda.is_bf16_supported()),
    "compute_capability": list(torch.cuda.get_device_capability(0)),
}
Path(sys.argv[1]).write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(info, ensure_ascii=False, indent=2))
""", encoding="utf-8")
run_py311(gpu_probe, ROOT / "gpu_info.json")
GPU_INFO = json.loads((ROOT / "gpu_info.json").read_text(encoding="utf-8"))
if GPU_INFO["gpu_vram_gib"] < 12:
    print("VRAM이 작습니다. 우선 10~30초, batch_size=1, LM 미사용을 유지하세요.")


## 4. 첫 실험 설정
`caption`은 음악의 장르·악기·분위기 설명입니다. BPM·길이는 별도 조건입니다.
BPM은 **목표값**이며 실제 음원의 BPM 측정 결과가 아닙니다. 같은 seed도 GPU·라이브러리가 달라지면 완전히 동일한 결과를 보장하지 않습니다.

`use_lm=True`는 선택 사항입니다. 0.6B LM을 추가로 다운로드하고 PyTorch 백엔드로 실행합니다.
첫 생성은 기본값을 유지하세요. `offload_to_cpu=True`는 일부 모델을 CPU에 두어 VRAM을 줄이는 설정이며 시스템 RAM과 전송 시간이 더 필요할 수 있습니다.


In [ ]:
EXPERIMENT = {
    "caption": (
        "Energetic electronic running music, steady clear drum beat, "
        "uplifting synth melody, consistent rhythm, instrumental, no vocals"
    ),
    "bpm": 170,
    "duration": 30.0,
    "seed": 42,
    "instrumental": True,
    "inference_steps": 8,
    "batch_size": 1,
    "audio_format": "wav",
    "use_lm": False,
    "lm_model": "acestep-5Hz-lm-0.6B",
    "offload_to_cpu": True,
}
assert 30 <= EXPERIMENT["bpm"] <= 300
assert 10 <= EXPERIMENT["duration"] <= 600
assert EXPERIMENT["batch_size"] == 1, "초안에서는 1개씩 생성하세요."
print(json.dumps(EXPERIMENT, ensure_ascii=False, indent=2))


## 5. 모델 다운로드
DiT·VAE·텍스트 인코더 등의 가중치를 공식 Hugging Face 저장소에서 받습니다.
공식 다운로드 검사가 기본 1.7B LM 폴더도 검사하므로 메인 모델 묶음을 다운로드합니다. **기본 실험에서는 그 LM을 메모리에 로드하지 않습니다.**

최초 실행 시 모델 저장소의 커밋도 기록합니다. 이후 같은 코랩 작업 폴더에서는 그 커밋을 재사용합니다.
모델은 실행 코드만으로 동작하지 않으며 이 다운로드가 완료되어야 합니다.


In [ ]:
download_script = ROOT / "download_models.py"
download_script.write_text("""
import json
import sys
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download

root = Path(sys.argv[1])
use_lm = sys.argv[2] == "true"
lm_model = sys.argv[3]
checkpoints = root / "ACE-Step-1.5" / "checkpoints"
checkpoints.mkdir(parents=True, exist_ok=True)
revision_file = root / "model_revisions.json"
revisions = json.loads(revision_file.read_text()) if revision_file.exists() else {}
api = HfApi()

def download(repo_id, target):
    if repo_id not in revisions:
        revisions[repo_id] = api.model_info(repo_id).sha
        revision_file.write_text(json.dumps(revisions, indent=2), encoding="utf-8")
    print(f"Downloading {repo_id} @ {revisions[repo_id]}", flush=True)
    snapshot_download(
        repo_id=repo_id,
        revision=revisions[repo_id],
        local_dir=str(target),
    )

download("ACE-Step/Ace-Step1.5", checkpoints)
if use_lm:
    download("ACE-Step/" + lm_model, checkpoints / lm_model)
print("모델 다운로드 완료", flush=True)
""", encoding="utf-8")
run_py311(
    download_script, ROOT,
    str(EXPERIMENT["use_lm"]).lower(),
    EXPERIMENT["lm_model"],
)


## 6. 재사용 가능한 추론 스크립트 만들기
다음 셀은 `infer_music.py`를 작성합니다. 이후 SageMaker용 컨테이너에서 모델 로딩·생성 부분을 재사용할 수 있습니다.

한 프로세스에서 모델을 한 번 로드한 뒤 요청 목록을 순서대로 처리합니다.
메타데이터에는 입력값, 실제 길이·샘플레이트, 모델 로딩 시간, 생성 시간, PyTorch GPU 메모리 피크, 코드·모델 버전을 기록합니다.
메모리 피크는 해당 프로세스의 PyTorch 할당·예약량이며 GPU 전체 사용량과는 다릅니다.


In [ ]:
INFERENCE_SCRIPT = ROOT / "infer_music.py"
INFERENCE_SCRIPT.write_text("import argparse\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom dataclasses import asdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nos.environ[\"TOKENIZERS_PARALLELISM\"] = \"false\"\n\nimport soundfile as sf\nimport torch\nfrom acestep.handler import AceStepHandler\nfrom acestep.inference import GenerationParams, GenerationConfig, generate_music\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--request\", required=True)\n    parser.add_argument(\"--output\", required=True)\n    args = parser.parse_args()\n    request_path = Path(args.request).resolve()\n    payload = json.loads(request_path.read_text(encoding=\"utf-8\"))\n    settings = payload[\"settings\"]\n    jobs = payload[\"jobs\"]\n    if not jobs:\n        raise ValueError(\"생성 요청이 없습니다.\")\n    if sys.version_info[:2] != (3, 11):\n        raise RuntimeError(\"추론용 Python 3.11이 필요합니다.\")\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"CUDA GPU가 필요합니다.\")\n\n    project_root = Path(__file__).resolve().parent / \"ACE-Step-1.5\"\n    checkpoints = project_root / \"checkpoints\"\n    os.environ[\"ACESTEP_CHECKPOINTS_DIR\"] = str(checkpoints)\n    output = Path(args.output).resolve()\n    output.mkdir(parents=True, exist_ok=True)\n\n    # 상태 반환은 공식 API의 (status_message, success) 순서입니다.\n    started = time.perf_counter()\n    dit = AceStepHandler()\n    status, success = dit.initialize_service(\n        project_root=str(project_root),\n        config_path=\"acestep-v15-turbo\",\n        device=\"cuda\",\n        use_flash_attention=False,\n        compile_model=False,\n        quantization=None,\n        offload_to_cpu=bool(settings[\"offload_to_cpu\"]),\n        offload_dit_to_cpu=False,\n        prefer_source=\"huggingface\",\n    )\n    print(status, flush=True)\n    if not success:\n        raise RuntimeError(\"DiT 초기화 실패: \" + status)\n    if str(dit.device) != \"cuda\":\n        raise RuntimeError(\"CUDA 요청이 CPU 등으로 전환되었습니다.\")\n\n    lm = None\n    use_lm = bool(settings[\"use_lm\"])\n    if use_lm:\n        from acestep.llm_inference import LLMHandler\n        lm = LLMHandler()\n        status, success = lm.initialize(\n            checkpoint_dir=str(checkpoints),\n            lm_model_path=settings[\"lm_model\"],\n            backend=\"pt\",\n            device=\"cuda\",\n            offload_to_cpu=bool(settings[\"offload_to_cpu\"]),\n            dtype=dit.dtype,\n        )\n        print(status, flush=True)\n        if not success:\n            raise RuntimeError(\"LM 초기화 실패: \" + status)\n    torch.cuda.synchronize()\n    loading_seconds = time.perf_counter() - started\n    source_commit = subprocess.check_output(\n        [\"git\", \"rev-parse\", \"HEAD\"], cwd=project_root, text=True\n    ).strip()\n    revision_file = project_root.parent / \"model_revisions.json\"\n    revisions = json.loads(revision_file.read_text()) if revision_file.exists() else {}\n    lock_path = project_root / \"uv.lock\"\n    import hashlib\n    lock_sha256 = hashlib.sha256(lock_path.read_bytes()).hexdigest()\n\n    manifest = {\n        \"created_at_utc\": datetime.now(timezone.utc).isoformat(),\n        \"source_commit\": source_commit,\n        \"model_revisions\": revisions,\n        \"uv_lock_sha256\": lock_sha256,\n        \"python\": sys.version,\n        \"torch\": torch.__version__,\n        \"torch_cuda\": torch.version.cuda,\n        \"gpu\": torch.cuda.get_device_name(0),\n        \"dtype\": str(dit.dtype),\n        \"model_loading_seconds\": loading_seconds,\n        \"settings\": settings,\n        \"runs\": [],\n    }\n    manifest_path = output / \"manifest.json\"\n    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n\n    for index, job in enumerate(jobs):\n        if not 30 <= int(job[\"bpm\"]) <= 300:\n            raise ValueError(\"BPM은 30~300 범위여야 합니다.\")\n        if not 10 <= float(job[\"duration\"]) <= 600:\n            raise ValueError(\"duration은 10~600초 범위여야 합니다.\")\n        run_dir = output / f\"run_{index:02d}\"\n        run_dir.mkdir(exist_ok=True)\n        params = GenerationParams(\n            task_type=\"text2music\",\n            caption=job[\"caption\"],\n            lyrics=\"[Instrumental]\" if job[\"instrumental\"] else job.get(\"lyrics\", \"\"),\n            instrumental=bool(job[\"instrumental\"]),\n            bpm=int(job[\"bpm\"]),\n            duration=float(job[\"duration\"]),\n            timesignature=\"4\",\n            seed=int(job[\"seed\"]),\n            inference_steps=int(settings[\"inference_steps\"]),\n            thinking=use_lm,\n            use_cot_metas=False,\n            use_cot_caption=False,\n            use_cot_lyrics=False,\n            use_cot_language=False,\n        )\n        config = GenerationConfig(\n            batch_size=1,\n            use_random_seed=False,\n            seeds=[int(job[\"seed\"])],\n            audio_format=settings[\"audio_format\"],\n        )\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n        torch.cuda.synchronize()\n        started = time.perf_counter()\n        with torch.inference_mode():\n            result = generate_music(dit, lm, params, config, save_dir=str(run_dir))\n        torch.cuda.synchronize()\n        generation_seconds = time.perf_counter() - started\n        if not result.success:\n            raise RuntimeError(result.error or result.status_message)\n        if not result.audios:\n            raise RuntimeError(\"음원 결과가 없습니다.\")\n        audio_records = []\n        for audio in result.audios:\n            # 성공 플래그가 있어도 음원 파일 저장이 실패할 수 있으므로 별도로 검사합니다.\n            raw_path = audio.get(\"path\")\n            if not raw_path:\n                raise RuntimeError(\"추론 결과의 음원 파일 경로가 비어 있습니다.\")\n            audio_path = Path(raw_path).resolve()\n            if not audio_path.is_file() or audio_path.stat().st_size == 0:\n                raise RuntimeError(\"음원 파일이 없거나 비어 있습니다.\")\n            info = sf.info(str(audio_path))\n            audio_records.append({\n                \"path\": str(audio_path),\n                \"relative_path\": str(audio_path.relative_to(output)),\n                \"sample_rate\": int(info.samplerate),\n                \"channels\": int(info.channels),\n                \"actual_duration_seconds\": float(info.duration),\n                \"bytes\": audio_path.stat().st_size,\n                \"model_reported_seed\": audio.get(\"params\", {}).get(\"seed\"),\n            })\n        record = {\n            \"index\": index,\n            \"request\": job,\n            \"effective_generation_params\": asdict(params),\n            \"generation_seconds\": generation_seconds,\n            \"peak_torch_allocated_gib\": torch.cuda.max_memory_allocated() / 2**30,\n            \"peak_torch_reserved_gib\": torch.cuda.max_memory_reserved() / 2**30,\n            \"audios\": audio_records,\n            \"status_message\": result.status_message,\n        }\n        manifest[\"runs\"].append(record)\n        manifest_path.write_text(\n            json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding=\"utf-8\"\n        )\n        print(f\"완료: run_{index:02d}, 생성 {generation_seconds:.2f}초\", flush=True)\n        # 이전 결과 텐서가 다음 생성의 메모리를 차지하지 않게 해제합니다.\n        del result\n    print(\"Manifest:\", manifest_path, flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
print("추론 스크립트 저장:", INFERENCE_SCRIPT)


## 7. 음악 생성 실행
이 셀을 다시 실행하면 새로운 결과 폴더를 만듭니다. 별도 프로세스이므로 매 실행마다 모델을 다시 로드합니다.
`generation_seconds`에는 모델 로딩·다운로드가 포함되지 않습니다. `wall_seconds`는 이번 프로세스 실행 전체 시간입니다.


In [ ]:
import time
from datetime import datetime

def make_job(bpm=None, seed=None, duration=None):
    return {
        "caption": EXPERIMENT["caption"],
        "bpm": EXPERIMENT["bpm"] if bpm is None else bpm,
        "duration": EXPERIMENT["duration"] if duration is None else duration,
        "seed": EXPERIMENT["seed"] if seed is None else seed,
        "instrumental": EXPERIMENT["instrumental"],
    }

def execute_jobs(jobs):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    result_dir = OUTPUT_ROOT / stamp
    result_dir.mkdir(parents=True)
    payload = {"settings": EXPERIMENT.copy(), "jobs": jobs}
    request_file = result_dir / "request.json"
    request_file.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    started = time.perf_counter()
    try:
        run_py311(
            INFERENCE_SCRIPT, "--request", request_file,
            "--output", result_dir,
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "추론이 실패했습니다. 위 로그를 확인하세요. CUDA OOM이면 길이를 10~30초로 "
            "줄이고 LM을 끈 뒤 재실행하세요. 실패 요청은 결과 폴더에 남습니다."
        ) from exc
    manifest_path = result_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest["wall_seconds"] = time.perf_counter() - started
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return result_dir, manifest

RESULT_DIR, MANIFEST = execute_jobs([make_job()])
print("결과:", RESULT_DIR)
print("모델 로딩:", round(MANIFEST["model_loading_seconds"], 2), "초")
print("프로세스 전체:", round(MANIFEST["wall_seconds"], 2), "초")


## 8. 결과 확인과 음악 재생
노트북 커널에서 파일을 재생합니다. 추론 라이브러리를 코랩 커널에 다시 설치할 필요는 없습니다.
목표 BPM과 실제 음원 BPM 일치 여부, 무보컬 여부, 박자 안정성은 별도로 평가하세요.


In [ ]:
from IPython.display import Audio, display

for run in MANIFEST["runs"]:
    print(
        f"목표 BPM={run['request']['bpm']} / seed={run['request']['seed']} / "
        f"생성={run['generation_seconds']:.2f}초 / "
        f"PyTorch 피크 할당={run['peak_torch_allocated_gib']:.2f}GiB"
    )
    for audio in run["audios"]:
        print(
            f"실제 길이={audio['actual_duration_seconds']:.2f}초, "
            f"샘플레이트={audio['sample_rate']}Hz, 채널={audio['channels']}"
        )
        display(Audio(filename=audio["path"]))


## 9. 선택: 같은 모델 로딩으로 BPM 비교
첫 음원 생성이 성공한 뒤 실행하세요. 비교 셀은 기본적으로 꺼져 있습니다.
170/85 BPM은 절반·두 배 박자 해석을 비교하려는 예시이며, 특정 케이던스에 적합한 템포라는 결론은 아닙니다.



















In [ ]:
RUN_COMPARISON = False  # 비교하려면 True로 변경

if RUN_COMPARISON:
    RESULT_DIR, MANIFEST = execute_jobs([
        make_job(bpm=170, seed=42),
        make_job(bpm=85, seed=42),
        make_job(bpm=170, seed=43),
    ])
    for run in MANIFEST["runs"]:
        print(run["request"], "생성 시간:", round(run["generation_seconds"], 2), "초")
        display(Audio(filename=run["audios"][0]["path"]))
else:
    print("비교 실험을 건너뜁니다. 현재 결과 폴더:", RESULT_DIR)


## 10. 결과·코드 ZIP 다운로드
현재 `RESULT_DIR`의 음원, 입력값, 기록과 추론 스크립트를 함께 저장합니다.
`uv.lock`·코드 커밋·모델 커밋·GPU 환경을 포함합니다. 모델 가중치 자체는 용량이 커서 ZIP에 넣지 않습니다.
코랩 다운로드 권한 요청이 표시되면 허용하세요.


In [ ]:
# from google.colab import files

# shutil.copy2(INFERENCE_SCRIPT, RESULT_DIR / "infer_music.py")
# shutil.copy2(REPO / "uv.lock", RESULT_DIR / "uv.lock")
# shutil.copy2(REPO / "pyproject.toml", RESULT_DIR / "pyproject.toml")
# shutil.copy2(ROOT / "gpu_info.json", RESULT_DIR / "gpu_info.json")
# shutil.copy2(ROOT / "model_revisions.json", RESULT_DIR / "model_revisions.json")
# environment_text = subprocess.check_output(
#     [UV, "pip", "freeze", "--python", str(PY311)], cwd=REPO, text=True
# )
# (RESULT_DIR / "environment.txt").write_text(environment_text, encoding="utf-8")
# archive_path = Path(shutil.make_archive(str(RESULT_DIR), "zip", root_dir=RESULT_DIR))
# print("ZIP:", archive_path)
# files.download(str(archive_path))
import json
import shutil
import subprocess
from pathlib import Path

RESULT_DIR = Path(RESULT_DIR)
manifest_file = RESULT_DIR / "manifest.json"
if not manifest_file.is_file():
    raise FileNotFoundError(f"생성 결과 기록이 없습니다: {manifest_file}")
MANIFEST = json.loads(manifest_file.read_text(encoding="utf-8"))

# 점검·버전 기록은 보조 파일입니다. 없어도 음원 ZIP을 만들 수 있습니다.
optional_files = [
    Path(INFERENCE_SCRIPT), REPO / "uv.lock", REPO / "pyproject.toml",
    ROOT / "gpu_info.json", ROOT / "model_revisions.json",
]
missing_files = []
for source in optional_files:
    if source.is_file():
        destination = RESULT_DIR / source.name
        if source.resolve() != destination.resolve():
            shutil.copy2(source, destination)
    else:
        missing_files.append(str(source))
        print("보조 파일 없음 — 건너뜀:", source.name)

environment_error = None
try:
    environment_text = subprocess.check_output(
        [str(UV), "pip", "freeze", "--python", str(PY311)],
        cwd=REPO, text=True, stderr=subprocess.STDOUT,
    )
    (RESULT_DIR / "environment.txt").write_text(environment_text, encoding="utf-8")
except (OSError, subprocess.CalledProcessError) as exc:
    environment_error = str(exc)
    print("패키지 목록 기록 실패 — 음원 저장은 계속합니다:", exc)

(RESULT_DIR / "export_info.json").write_text(
    json.dumps({"missing_optional_files": missing_files,
                "environment_capture_error": environment_error},
               ensure_ascii=False, indent=2), encoding="utf-8",
)
archive_path = Path(shutil.make_archive(str(RESULT_DIR), "zip", root_dir=RESULT_DIR))
print("ZIP 준비 완료:", archive_path)

DOWNLOAD_ZIP = False  # 지정한 Drive 폴더에 저장합니다. PC 다운로드도 원하면 True.
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(archive_path))


## 11. 선택: Google Drive에 결과 ZIP 백업
본인이 Drive 인증을 진행하는 선택 셀입니다. 기본값은 꺼져 있습니다.
모델을 Drive에서 직접 읽도록 구성하지 않았습니다. 현재 결과 ZIP만 복사합니다.


In [ ]:
# SAVE_TO_DRIVE = False  # 백업하려면 True로 변경

# if SAVE_TO_DRIVE:
#     from google.colab import drive
#     drive.mount("/content/drive")
#     backup_dir = Path("/content/drive/MyDrive/Runiverse/ACE-Step-1.5/results")
#     backup_dir.mkdir(parents=True, exist_ok=True)
#     destination = backup_dir / archive_path.name
#     shutil.copy2(archive_path, destination)
#     print("백업 완료:", destination)
# else:
#     print("Drive 백업을 건너뜁니다.")
import hashlib
import json
import mimetypes
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

DRIVE_FOLDER_ID = "1l4inc6avgcnbPGV_xgAZb17rMDigRowr"
RESULT_DIR = Path(RESULT_DIR)
MANIFEST = json.loads((RESULT_DIR / "manifest.json").read_text(encoding="utf-8"))

# 업로드 전 음원 존재 여부를 검사합니다. ZIP 셀과 독립적으로 실행할 수 있습니다.
stamp = datetime.fromisoformat(
    MANIFEST["created_at_utc"].replace("Z", "+00:00")
).astimezone(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S_%f_KST")
uploads = []
for run_index, run in enumerate(MANIFEST["runs"], start=1):
    request = run["request"]
    for audio_index, audio in enumerate(run["audios"], start=1):
        # 상대 경로로 찾아서 결과 폴더를 다른 위치로 옮겨도 사용할 수 있습니다.
        source = (RESULT_DIR / audio["relative_path"] if audio.get("relative_path")
                  else Path(audio["path"]))
        if not source.is_file() or source.stat().st_size == 0:
            raise FileNotFoundError(f"음원 파일이 없거나 비어 있습니다: {source}")
        title = (
            f"Runiverse_{stamp}_BPM{int(request['bpm'])}_"
            f"{float(request['duration']):g}s_seed{int(request['seed'])}_"
            f"r{run_index:02d}_a{audio_index:02d}{source.suffix.lower()}"
        )
        uploads.append((source, title))
if not uploads:
    raise RuntimeError("저장할 음원이 없습니다. 먼저 음악 생성 셀을 실행하세요.")

uploads.append((RESULT_DIR / "manifest.json", f"Runiverse_{stamp}_manifest.json"))
current_archive = RESULT_DIR.with_suffix(".zip")
if current_archive.is_file():
    uploads.append((current_archive, f"Runiverse_{stamp}_results.zip"))

auth.authenticate_user()  # 대상 폴더에 파일을 추가할 수 있는 계정으로 인증하세요.
credentials, _ = google.auth.default()
drive_service = build("drive", "v3", credentials=credentials, cache_discovery=False)
folder = drive_service.files().get(
    fileId=DRIVE_FOLDER_ID, fields="id,name,mimeType,capabilities(canAddChildren)",
    supportsAllDrives=True,
).execute()
if folder["mimeType"] != "application/vnd.google-apps.folder":
    raise RuntimeError("저장 대상이 Drive 폴더가 아닙니다.")
if folder.get("capabilities", {}).get("canAddChildren") is False:
    raise PermissionError("현재 인증 계정에는 이 폴더에 파일을 추가할 권한이 없습니다.")
print("저장 대상 폴더:", folder["name"])

saved = []
for source, title in uploads:
    # 같은 결과로 셀을 재실행해도 동일한 파일을 중복 생성하지 않습니다.
    with source.open("rb") as content_file:
        content_md5 = hashlib.file_digest(content_file, "md5").hexdigest()
    upload_key = hashlib.sha256(
        f"{RESULT_DIR.name}/{title}".encode("utf-8")
    ).hexdigest()
    matches = drive_service.files().list(
        q=(f"'{DRIVE_FOLDER_ID}' in parents and trashed = false and "
           "appProperties has { key='runiverse_upload_key' and "
           f"value='{upload_key}' }}"),
        fields="files(id,name,parents,size,md5Checksum,webViewLink)",
        pageSize=100, supportsAllDrives=True, includeItemsFromAllDrives=True,
    ).execute().get("files", [])
    matching = next((item for item in matches
                     if item.get("md5Checksum") == content_md5), None)
    if matching is not None:
        uploaded = matching
        print("이미 저장됨:", title)
    else:
        uploaded = drive_service.files().create(
            body={"name": title, "parents": [DRIVE_FOLDER_ID],
                  "appProperties": {"runiverse_upload_key": upload_key}},
            media_body=MediaFileUpload(
                str(source), mimetype=mimetypes.guess_type(title)[0]
                or "application/octet-stream", resumable=True,
            ),
            fields="id", supportsAllDrives=True,
        ).execute(num_retries=3)
        uploaded = drive_service.files().get(
            fileId=uploaded["id"], supportsAllDrives=True,
            fields="id,name,parents,size,md5Checksum,webViewLink",
        ).execute()
        print("저장 완료:", uploaded["name"])
    if (DRIVE_FOLDER_ID not in uploaded.get("parents", [])
            or int(uploaded.get("size", -1)) != source.stat().st_size
            or uploaded.get("md5Checksum") != content_md5):
        raise RuntimeError(f"업로드한 파일의 폴더·크기·내용 확인 실패: {title}")
    if uploaded.get("webViewLink"):
        print(uploaded["webViewLink"])
    saved.append({"local_path": str(source), **uploaded})

(RESULT_DIR / "drive_uploads.json").write_text(
    json.dumps(saved, ensure_ascii=False, indent=2), encoding="utf-8",
)
print(f"총 {len(saved)}개 파일 저장 확인 완료")


In [ ]:
import zipfile
from google.colab import files

backup_path = ROOT / f"Runiverse_{stamp}_audio_backup.zip"

with zipfile.ZipFile(
    backup_path, "w", compression=zipfile.ZIP_DEFLATED
) as bundle:
    for source, title in uploads:
        if source.suffix.lower() != ".zip":
            bundle.write(source, arcname=title)

files.download(str(backup_path))

## 문제가 생겼을 때

| 상황 | 확인·조치 |
|---|---|
| GPU가 없다는 오류 | 코랩 GPU 런타임 선택 후 1번 셀부터 실행 |
| Python 버전 오류 | 2번 셀의 별도 가상환경 설치 성공 여부 확인 |
| CUDA 드라이버 오류 | 1·3번 출력과 함께 코랩 드라이버/설치된 PyTorch CUDA 호환성 확인. 시스템 CUDA나 드라이버를 임의로 덮어쓰지 않기 |
| 설치 중 디스크 부족 | 남은 디스크 확인. 모델과 PyTorch 설치 공간이 함께 필요 |
| Hugging Face 다운로드 실패 | 네트워크·저장 공간 확인 후 5번 셀 재실행 |
| CUDA out of memory | LM 끄기, 길이 10~30초, 생성 1개 유지. 다른 GPU 작업 종료 |
| 생성 성공인데 파일 없음 | 스크립트가 파일 저장까지 검사하므로 해당 오류·로그 확인 |
| 생성이 느림 | 최초 모델 로딩과 생성 시간을 구분. CPU offload와 GPU 종류 확인 |
| BPM 또는 보컬이 기대와 다름 | 실제 음원을 듣고 측정. 입력 조건은 결과 보장이 아니므로 여러 seed·설명 비교 |
| 코랩 런타임이 삭제됨 | 설치·다운로드 셀부터 재실행. 저장한 노트북·ZIP을 활용 |

**SageMaker로 옮길 때:** 추론 함수를 재사용하고 모델 로딩을 서버 시작 시 한 번 수행하도록 변경합니다. 8080 포트의 `/ping`·`/invocations`, S3 출력, 작업 상태 관리는 별도 구현 대상입니다. 이 노트북은 AWS 배포를 실행하지 않습니다.

공식 참고:
- [ACE-Step 1.5 저장소](https://github.com/ace-step/ACE-Step-1.5)
- [고정 버전 Python 추론 API](https://github.com/ace-step/ACE-Step-1.5/blob/ca1e85fe9430179831e6bc6be790c332190a3866/docs/en/INFERENCE.md)
- [고정 버전 설치 안내](https://github.com/ace-step/ACE-Step-1.5/blob/ca1e85fe9430179831e6bc6be790c332190a3866/docs/en/INSTALL.md)
- [Colab FAQ](https://research.google.com/colaboratory/faq.html)

작성일: 2026-10-06. 코드·모델의 라이선스 원문은 실제 사용하는 고정 버전에서 함께 보관하세요.
